# 🏯 Канон Сянься — Выделенный сервер перевода и Автономный ночной воркер на Google Colab L4 (24 ГБ VRAM)

Этот ноутбук работает в двух режимах на вашей карте **NVIDIA L4 (24 GB)** в рамках подписки **Google AI Ultra**:
1. **Интерактивный сервер для веб-студии (Ячейка 1):** поднимает HTTPS-туннель Cloudflare к локальному Ollama (`Gemma 4 31B`, `Gemma 4 26B A4B MoE`, `TranslateGemma 27B`, `Qwen 3 32B`).
2. **Автономный ночной пакетный воркер через Google Drive (Ячейка 2):** монтирует ваш Google Диск (`/content/drive/MyDrive/KanonXianxia/kanon_project_bundle.json`), переводит все непереведённые главы в **2 честных прохода** (Проход 1: Точность и Канон -> Проход 2: Хирургический редактор по протоколу `Perevod_Ready`) и атомарно сохраняет чекпоинт после каждой главы. Компьютер на ночь можно выключать!

In [ ]:
#@title 🚀 1. Выберите модель и запустите сервер на L4 (24 ГБ VRAM)
MODEL_ID = "gemma4:26b-a4b" #@param ["gemma4:26b-a4b", "gemma4:31b", "translategemma:27b", "qwen3:32b", "mistral-small3.1:24b"] {allow-input: true}
CTX_LEN = 16384 #@param [8192, 16384, 32768] {type:"raw"}

import subprocess, time, re, os

print("🔍 Проверка GPU...")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

print("\n📦 1/4 Установка Ollama и Cloudflare Tunnel...")
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# Оптимизация под NVIDIA L4 (24 GB VRAM): Flash Attention + сжатие KV-кэша q8_0
os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
os.environ["OLLAMA_ORIGINS"] = "*"
os.environ["OLLAMA_FLASH_ATTENTION"] = "1"
os.environ["OLLAMA_KV_CACHE_TYPE"] = "q8_0"
os.environ["OLLAMA_NUM_PARALLEL"] = "2"
os.environ["OLLAMA_KEEP_ALIVE"] = "24h"
os.environ["OLLAMA_CONTEXT_LENGTH"] = str(CTX_LEN)

print("\n⚡ 2/4 Запуск фонового демона Ollama...")
ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)
time.sleep(5)

print(f"\n🧠 3/4 Загрузка весов {MODEL_ID} в видеопамять L4...")
!ollama pull {MODEL_ID}

print("\n🌐 4/4 Поднятие защищённого HTTPS-туннеля Cloudflare...")
tunnel_proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://localhost:11434", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for _ in range(60):
    line = tunnel_proc.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

print("\n" + "="*76)
print("✅ СЕРВЕР COLAB L4 ГОТОВ К РАБОТЕ!")
print(f"🔗 Скопируйте этот URL в окно «Colab L4 / Модели» в студии: {public_url}")
print(f"🧠 Активная модель: {MODEL_ID} (Контекст: {CTX_LEN}, KV-Cache: q8_0)")
print("="*76)


In [ ]:
#@title 🌙 2. Автономный ночной воркер через Google Drive (компьютер можно выключить)
DRIVE_FOLDER = "/content/drive/MyDrive/KanonXianxia" #@param {type:"string"}
BUNDLE_FILENAME = "kanon_project_bundle.json" #@param {type:"string"}
TWO_PASS_SURGICAL = True #@param {type:"boolean"}

import json, os, re, time, urllib.request
from google.colab import drive

drive.mount('/content/drive')
os.makedirs(DRIVE_FOLDER, exist_ok=True)
bundle_path = os.path.join(DRIVE_FOLDER, BUNDLE_FILENAME)

if not os.path.exists(bundle_path):
    print(f"⚠️ Файл {bundle_path} пока не найден.")
    print("В веб-студии нажмите «Colab L4» -> «Скачать бандл для ночного Google Drive» и положите его в папку KanonXianxia на вашем Google Диске.")
else:
    with open(bundle_path, 'r', encoding='utf-8') as f:
        bundle = json.load(f)

    project = bundle.get('project', bundle)
    chapters = project.get('chapters', [])
    glossary = project.get('glossary', {})
    style_guide = project.get('styleGuideMd', '')
    active_model = bundle.get('llmConfig', {}).get('modelId', MODEL_ID)

    def ollama_chat(sys_prompt, user_prompt, temp=0.2):
        req = urllib.request.Request(
            'http://localhost:11434/api/chat',
            data=json.dumps({
                'model': active_model,
                'stream': False,
                'format': 'json',
                'options': {'temperature': temp, 'num_ctx': CTX_LEN},
                'messages': [
                    {'role': 'system', 'content': sys_prompt},
                    {'role': 'user', 'content': user_prompt}
                ]
            }).encode('utf-8'),
            headers={'Content-Type': 'application/json'}
        )
        with urllib.request.urlopen(req, timeout=240) as resp:
            body = json.loads(resp.read().decode('utf-8'))
            raw = body.get('message', {}).get('content', '{}')
            raw = re.sub(r'<think>[\s\S]*?</think>', '', raw).strip()
            return json.loads(raw)

    def apply_surgical_patches(text, patches):
        applied = 0
        for p in (patches or []):
            find_s = (p.get('find') or '').strip()
            rep_s = (p.get('replace') or '').strip()
            if not find_s or find_s == rep_s:
                continue
            count = text.count(find_s)
            if count == 1 or (count > 1 and p.get('all')):
                text = text.replace(find_s, rep_s)
                applied += 1
        return text, applied

    pending = [c for c in chapters if not c.get('translated') or not (c.get('rusText') or '').strip()]
    print(f"📚 Найдено глав для ночного перевода: {len(pending)} из {len(chapters)} (Модель: {active_model})")

    for idx, ch in enumerate(chapters):
        if ch.get('translated') and (ch.get('rusText') or '').strip():
            continue
        t0 = time.time()
        num = ch.get('number', idx + 1)
        title_eng = ch.get('titleEng', f'Chapter {num}')
        eng_text = ch.get('engText', '')

        sys_p1 = 'You are Pass 1 Semantic & Canon Translator for Russian Xianxia/LitRPG. Translate 1:1 paragraph-by-paragraph, preserve all Arabic numbers and --- dividers, follow the glossary, and return JSON: {"titleRus": "...", "rusText": "...", "summary": "..."}'
        usr_p1 = f"Glossary:\n{json.dumps(glossary, ensure_ascii=False)}\n\nStyle Guide:\n{style_guide}\n\nChapter {num} ({title_eng}):\n{eng_text}"
        res_p1 = ollama_chat(sys_p1, usr_p1, temp=0.2)
        rus_text = res_p1.get('rusText', '')
        title_rus = res_p1.get('titleRus', f'Глава {num}')

        patches_applied = 0
        if TWO_PASS_SURGICAL and rus_text:
            try:
                sys_p2 = 'Ты — литературный редактор (Проход 2, Surgical Patching). Найди в русском черновике кальки («в своём сердце», «свои глаза», «горько улыбнулся») и верни JSON: {"patches": [{"find": "точная цитата", "replace": "исправленный вариант", "reason": "...", "all": false}]}'
                usr_p2 = f"ORIGINAL ENG:\n{eng_text}\n\nDRAFT RUS:\n{rus_text}"
                res_p2 = ollama_chat(sys_p2, usr_p2, temp=0.35)
                rus_text, patches_applied = apply_surgical_patches(rus_text, res_p2.get('patches', []))
            except Exception as e:
                print(f"  [Pass 2 пропущен для Гл. {num}: {e}]")

        ch['titleRus'] = title_rus
        ch['rusText'] = rus_text
        ch['summary'] = res_p1.get('summary', '')
        ch['translated'] = True
        ch['edited'] = True
        ch['wordCountRus'] = len(rus_text.split())

        # Атомарное сохранение чекпоинта на Google Drive после КАЖДОЙ главы
        tmp_path = bundle_path + '.tmp'
        with open(tmp_path, 'w', encoding='utf-8') as wf:
            json.dump(bundle, wf, ensure_ascii=False, indent=2)
        os.replace(tmp_path, bundle_path)
        print(f"✅ Глава {num} («{title_rus}») переведена за {time.time() - t0:.1f} сек · Хирургических патчей Pass 2: {patches_applied} · Чекпоинт сохранён на Google Drive!")

    print("🎉 Ночной пакетный перевод полностью завершён! Загрузите обновлённый kanon_project_bundle.json обратно в студию.")
